In [ ]:
import csv
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

# url = "https://www.reddit.com/r/slavelabour/new/"
url = "https://www.reddit.com/r/forhire/new/"

# Specify the subreddit URL
subreddit_url = url

# Set up the Chrome WebDriver
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
# options.headless = True

options.add_argument("--headless")  # Run Chrome in headless mode
options.add_argument("--disable-gpu")  # Disable GPU acceleration (optional but useful in headless mode)
options.add_argument("--no-sandbox")  # Recommended for headless mode in some environments
options.add_argument("--disable-dev-shm-usage")  # Overcome limited resource problems
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/92.0.4515.159 Safari/537.36")


driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
driver.get(subreddit_url)

# Accept Reddit's cookies if prompted (optional)
try:
    cookies_button = driver.find_element(By.XPATH, "//button[contains(text(), 'Accept All')]")
    cookies_button.click()
except:
    pass

# Scroll the page a few times to load more posts
scroll_pause_time = 2
scroll_count = 2

for _ in range(scroll_count):
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(scroll_pause_time)

# After scrolling, get the page source for Beautiful Soup
page_source = driver.page_source

# Close the Selenium driver
driver.quit()

# Parse the page source with Beautiful Soup
soup = BeautifulSoup(page_source, "html.parser")

# List to store the scraped data
scraped_data = []

# Find all post containers
posts = soup.find_all(id=lambda x: x and x.startswith("post-title-t3_"))

# Loop through each post container and extract data
for post in posts:
    # Extract the unique post ID from the title's id attribute
    post_id = post.get("id").replace("post-title-", "")
    
    # Get the title
    title = post.get_text(strip=True)

    link_tag = post.find_previous("a", href=True)
    user_id_link = link_tag['href'] if link_tag else "N/A"
    
    # If the post_link is relative, add the base URL of Reddit
    if user_id_link != "N/A" and not user_id_link.startswith("http"):
        user_id_link = "https://www.reddit.com" + user_id_link

        
    # Find the flair using the extracted post ID
    # flair = soup.find(attrs={"post-id": post_id})
    # flair = soup.find("div", class_="flair-content [&_.flair-image]:align-bottom max-w-full overflow-hidden whitespace-nowrap text-ellipsis")
    # flairs = driver.find_elements(By.CLASS_NAME, "flair-content")
    # flair_text = flairs[0].text if flairs else "N/A"  # Adjust to get specific flair as needed
    # flair_text = flair.get_text(strip=True) if flair else "N/A"
    
    # Find the description content
    description_div = soup.find(id=f"{post_id}-post-rtjson-content")
    if description_div:
        paragraphs = description_div.find_all("p")
        description = "\n".join([p.get_text(strip=True) for p in paragraphs])
    else:
        description = "N/A"
    
    # Store the data in a dictionary
    post_data = {
        "Post ID": post_id,
        "UserID Link":user_id_link,
        "Title": title,
        "Flair": 'N/A Flair Text - Manually Added',
        "Description": description
    }
    
    scraped_data.append(post_data)


for i in scraped_data:
    # Get the title
    title = i['Title']
    
    # Use regex to identify and tag the flair
    if re.search(r'\[For Hire\]', title, re.IGNORECASE):
        i['Flair'] = "For Hire"
    elif re.search(r'\[Hiring\]', title, re.IGNORECASE):
        i['Flair'] = "Hiring"
    else:
        i['Flair'] = "Other"



In [ ]:
print(len(scraped_data))
for i in scraped_data:
    print(i['Post ID'])
    # print(i['Post Link'])
    print(i['UserID Link'])
    print(i['Flair'])
    print(i['Title'])
    print(i['Description'][:50])
    print('\n')
    print("="*40)
# 
# print(len(scraped_data))
# print(scraped_data[52])

53
t3_1gqs1wk
https://www.reddit.com/user/itsAvision/


KeyError: 'UserID Link'

# Keyoword Tagging

In [52]:
# Define the keywords for Graphic Design (GD)
graphic_design_keywords = r"(photoshop|illustrator|UI/UX|graphic design|typography|posts)"

for i in scraped_data:
    # Get the description of the job
    description = i.get('Description', '')

    # Use regex to identify if any graphic design keywords are present in the description
    if re.search(graphic_design_keywords, description, re.IGNORECASE):
        i['Tag'] = "GD (graphic design)"
    else:
        i['Tag'] = "Other"

# Print the results with tags
for i in scraped_data:
    print(f"Title: {i['Title']}")
    print(f"Description: {i['Description']}")
    print(f"Tag: {i['Tag']}")
    print("=" * 40)


Title: [Hiring] a webscrapper
Description: looking for someone to webscrape a website like truepeoplesearch for me.
Tag: Other
Title: [Hiring] Content Researcher
Description: Hello! We’re currently recruiting for aContentResearcherposition for our YouTube channel focused on scary/creepy stories.
Skilled at web research, with a knack for finding fresh clips that haven’t appeared on similar channels.
Knowledgeable about the topics, research methods, and clip criteria.
Able to source 1 video per day when possible (typically 30 clips per video).
For an idea of our content style, check out a similar channel here:Chills YouTube Channel
If you believe you’re a good fit, please message me here or contact me onTelegram: bmodyth.
This post will be removed once we find suitable candidates.
Tag: Other
Title: [FOR HIRE] I can edit your headshots and photos to a more professional level.
Description: Hello everyone!
I can elevate your headshots and photos to a more professional level.
My services inc

In [57]:
import re

# Define keyword patterns for each tag
graphic_design_keywords = r"(photoshop|illustrator|UI/UX|graphic design|typography|posts)"
developer_keywords = r"(software development|programming|full-stack|backend|frontend|JavaScript|C\+\+|Java|Python|React|Angular|Node\.js|APIs|cloud computing|Git|RESTful|object-oriented programming|data structures|algorithms|Agile|CI/CD)"
python_developer_keywords = r"(Python|Django|webscraper|webscrapper|webscrape|Flask|Pandas|NumPy|API development|REST APIs|FastAPI|object-oriented programming|data analysis|SQL|PostgreSQL|MySQL|MongoDB|machine learning|data science|ETL|testing|debugging|Git|unit testing|Docker|AWS|Azure|Lambda)"
social_media_manager_keywords = r"(social media marketing|content creation|strategy|analytics|branding|Instagram|Facebook|Twitter|LinkedIn|TikTok|SEO|Hootsuite|Sprout Social|Buffer|Engagement|hashtags|campaigns|community management|metrics|advertising|paid ads|audience targeting)"
video_editor_keywords = r"(video editing|Premiere Pro|After Effects|DaVinci Resolve|Final Cut Pro|motion graphics|color grading|visual effects|animation|storyboarding|YouTube|social media videos|audio editing|transitions|text overlays|video production|timelines|cutting|rendering|storytelling)"
mern_stack_developer_keywords = r"(MERN stack|MongoDB|Express\.js|React\.js|Node\.js|JavaScript|REST APIs|full-stack development|front-end|back-end|NoSQL|JWT authentication|React hooks|Redux|state management|MongoDB Atlas|Webpack|npm|Git|CSS|HTML|Agile)"

# Iterate over each post to assign tags
for i in scraped_data:
    description = i.get('Description', '')
    tags = []

    # Check for each set of keywords and add the relevant tag if it matches
    if re.search(graphic_design_keywords, description, re.IGNORECASE):
        tags.append("GD (graphic design)")
    if re.search(developer_keywords, description, re.IGNORECASE):
        tags.append("Developer")
    if re.search(python_developer_keywords, description, re.IGNORECASE):
        tags.append("Python Developer")
    if re.search(social_media_manager_keywords, description, re.IGNORECASE):
        tags.append("Social Media Manager")
    if re.search(video_editor_keywords, description, re.IGNORECASE):
        tags.append("Video Editor")
    if re.search(mern_stack_developer_keywords, description, re.IGNORECASE):
        tags.append("MERN Stack Developer")

    # Assign tags to the post
    i['Tags'] = tags if tags else ["Other"]

# Print the results with tags
for i in scraped_data:
    print(f"Title: {i['Title']}")
    print(f"Description: {i['Description']}")
    print(f"Tags: {', '.join(i['Tags'])}")
    print("=" * 40)


Title: [Hiring] a webscrapper
Description: looking for someone to webscrape a website like truepeoplesearch for me.
Tags: Python Developer
Title: [Hiring] Content Researcher
Description: Hello! We’re currently recruiting for aContentResearcherposition for our YouTube channel focused on scary/creepy stories.
Skilled at web research, with a knack for finding fresh clips that haven’t appeared on similar channels.
Knowledgeable about the topics, research methods, and clip criteria.
Able to source 1 video per day when possible (typically 30 clips per video).
For an idea of our content style, check out a similar channel here:Chills YouTube Channel
If you believe you’re a good fit, please message me here or contact me onTelegram: bmodyth.
This post will be removed once we find suitable candidates.
Tags: Video Editor
Title: [FOR HIRE] I can edit your headshots and photos to a more professional level.
Description: Hello everyone!
I can elevate your headshots and photos to a more professional le

In [58]:
import gspread
from google.auth.transport.requests import Request
from google.oauth2.service_account import Credentials

SERVICE_ACCOUNT_FILE = 'job-finder-441316-9a23f1d9e136.json'

# Define the scope for Google Sheets API
SCOPES = ['https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/drive.file']

# Authenticate and create a client
credentials = Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)

# Ensure the credentials are refreshed if necessary
if credentials.expired and credentials.refresh_token:
    credentials.refresh(Request())

# Authorize the credentials
client = gspread.authorize(credentials)

# Open your Google Sheet by name or by key
sheet = client.open_by_key("1WAIfQQM-5vsoWLVUlG9IyKH8TVwWjTMzCY40FuluuQc")  # or use .get_worksheet(0) for the first sheet

# print(client.sheet1.get('A1'))

worksheet = sheet.get_worksheet(0)



# Function to find the last empty row in the sheet
def find_last_row():
    row = 1
    while worksheet.cell(row, 1).value:  # Check if cell in column A is empty
        row += 1
    return row


# Function to add scraped data to Google Sheets
def add_scraped_data_to_sheet(scraped_data):
    last_row = find_last_row()
    
    for i, post in enumerate(scraped_data, start=last_row):
        # Insert data into columns based on structure
        # worksheet.update_cell(i, 1, post['ID'])                # Column A: Unique ID
        # worksheet.update_cell(i, 1, post['Link'])              # Column B: Link to post
        worksheet.update_cell(i, 2, post['Title'])             # Column C: Title
        worksheet.update_cell(i, 3, post['Description'])       # Column D: Description
        worksheet.update_cell(i, 4, ', '.join(post['Tags']))   # Column E: Tags (comma-separated)
        worksheet.update_cell(i, 5, post['Flair'])             # Column F: Flair

    print("Data added to Google Sheets.")

# Example of `scraped_data` structure
# scraped_data = [
#     {
#         "ID": "unique_post_id_1",
#         "Link": "https://example.com/post_link",
#         "Title": "Example Title",
#         "Description": "Example Description",
#         "Tags": ["Tag1", "Tag2"],
#         "Flair": "Example Flair"
#     },
#     ...
# ]
# Call the function with your scraped data
add_scraped_data_to_sheet(scraped_data)

KeyboardInterrupt: 